# Part 2: Chain-of-Thought (CoT) Training Data Generation

Generate step-by-step reasoning chains for all 9,500 training puzzles.
These will be used to fine-tune Nemotron-3-Nano-30B via LoRA.

**Strategy:**
- For categories we can solve programmatically (numeral, gravitational, unit conversion): Generate deterministic CoT
- For categories requiring pattern inference (bit manipulation, text encryption, transformation rules): Generate CoT templates that show the reasoning process
- All outputs end with `\boxed{answer}` to match evaluation format

In [1]:
import csv
import re
import json
import numpy as np
import pandas as pd
from collections import defaultdict

train_df = pd.read_csv('./train.csv')

def classify_puzzle(prompt):
    if 'bit manipulation' in prompt: return 'bit_manipulation'
    elif 'encryption' in prompt: return 'text_encryption'
    elif 'numeral system' in prompt: return 'numeral_system'
    elif 'unit conversion' in prompt: return 'unit_conversion'
    elif 'gravitational constant' in prompt: return 'gravitational'
    elif 'transformation rules' in prompt: return 'transformation_rules'
    return 'unknown'

train_df['category'] = train_df['prompt'].apply(classify_puzzle)
print(f'Total training puzzles: {len(train_df)}')
print(train_df['category'].value_counts())

Total training puzzles: 9500
category
bit_manipulation        1602
gravitational           1597
unit_conversion         1594
text_encryption         1576
numeral_system          1576
transformation_rules    1555
Name: count, dtype: int64


## 1. CoT Generators by Category

### 1.1 Numeral System (100% solvable)

In [2]:
def cot_numeral_system(prompt, answer):
    """Generate step-by-step reasoning for decimal to Roman numeral conversion."""
    match = re.search(r'write the number (\d+)', prompt)
    if not match:
        return None
    num = int(match.group(1))
    
    steps = []
    steps.append(f"I need to convert {num} to Roman numerals.")
    steps.append(f"")
    steps.append(f"Roman numeral values: M=1000, CM=900, D=500, CD=400, C=100, XC=90, L=50, XL=40, X=10, IX=9, V=5, IV=4, I=1")
    steps.append(f"")
    
    val = [1000, 900, 500, 400, 100, 90, 50, 40, 10, 9, 5, 4, 1]
    syms = ['M', 'CM', 'D', 'CD', 'C', 'XC', 'L', 'XL', 'X', 'IX', 'V', 'IV', 'I']
    
    remaining = num
    result_parts = []
    for v, s in zip(val, syms):
        if remaining >= v:
            count = remaining // v
            steps.append(f"{remaining} ÷ {v} = {count} remainder {remaining % v} → {s * count}")
            result_parts.append(s * count)
            remaining = remaining % v
    
    roman = ''.join(result_parts)
    steps.append(f"")
    steps.append(f"Combining: {' + '.join(result_parts)} = {roman}")
    steps.append(f"")
    steps.append(f"\\boxed{{{answer}}}")
    
    return '\n'.join(steps)

# Test
row = train_df[train_df['category'] == 'numeral_system'].iloc[0]
print(cot_numeral_system(row['prompt'], row['answer']))

I need to convert 38 to Roman numerals.

Roman numeral values: M=1000, CM=900, D=500, CD=400, C=100, XC=90, L=50, XL=40, X=10, IX=9, V=5, IV=4, I=1

38 ÷ 10 = 3 remainder 8 → XXX
8 ÷ 5 = 1 remainder 3 → V
3 ÷ 1 = 3 remainder 0 → III

Combining: XXX + V + III = XXXVIII

\boxed{XXXVIII}


### 1.2 Gravitational Constant

In [3]:
def cot_gravitational(prompt, answer):
    """Generate step-by-step reasoning for gravitational constant problems."""
    examples = re.findall(r'For t = ([\d.]+)s, distance = ([\d.]+) m', prompt)
    query_match = re.search(r'for t = ([\d.]+)s', prompt)
    if not examples or not query_match:
        return None

    t_query = float(query_match.group(1))
    
    steps = []
    steps.append(f"I need to find the falling distance for t = {t_query}s using d = 0.5*g*t².")
    steps.append(f"")
    steps.append(f"First, I'll determine g from the given examples using g = 2d/t²:")
    steps.append(f"")
    
    g_values = []
    for t_str, d_str in examples:
        t = float(t_str)
        d = float(d_str)
        g = 2 * d / (t * t)
        g_values.append(g)
        steps.append(f"t = {t_str}s, d = {d_str}m: g = 2×{d_str}/{t_str}² = {g:.4f}")
    
    g_avg = np.mean(g_values)
    steps.append(f"")
    steps.append(f"Average g ≈ {g_avg:.4f}")
    steps.append(f"")
    steps.append(f"For t = {t_query}s:")
    steps.append(f"d = 0.5 × {g_avg:.4f} × {t_query}² = 0.5 × {g_avg:.4f} × {t_query**2:.4f} = {0.5*g_avg*t_query**2:.2f}")
    steps.append(f"")
    steps.append(f"\\boxed{{{answer}}}")
    
    return '\n'.join(steps)

# Test
row = train_df[train_df['category'] == 'gravitational'].iloc[0]
print(cot_gravitational(row['prompt'], row['answer']))

I need to find the falling distance for t = 4.41s using d = 0.5*g*t².

First, I'll determine g from the given examples using g = 2d/t²:

t = 1.37s, d = 14.92m: g = 2×14.92/1.37² = 15.8986
t = 4.27s, d = 144.96m: g = 2×144.96/4.27² = 15.9009
t = 3.28s, d = 85.54m: g = 2×85.54/3.28² = 15.9020
t = 3.67s, d = 107.09m: g = 2×107.09/3.67² = 15.9018
t = 1.78s, d = 25.19m: g = 2×25.19/1.78² = 15.9008

Average g ≈ 15.9008

For t = 4.41s:
d = 0.5 × 15.9008 × 4.41² = 0.5 × 15.9008 × 19.4481 = 154.62

\boxed{154.62}


### 1.3 Unit Conversion

In [4]:
def cot_unit_conversion(prompt, answer):
    """Generate step-by-step reasoning for unit conversion."""
    examples = re.findall(r'([\d.]+) m becomes ([\d.]+)', prompt)
    query_match = re.search(r'convert the following measurement: ([\d.]+) m', prompt)
    if not examples or not query_match:
        return None

    x_query = float(query_match.group(1))
    
    steps = []
    steps.append(f"I need to convert {x_query} m using the secret conversion rule.")
    steps.append(f"")
    steps.append(f"Let me find the conversion factor from the examples:")
    steps.append(f"")
    
    factors = []
    for x_str, y_str in examples:
        x = float(x_str)
        y = float(y_str)
        f = y / x
        factors.append(f)
        steps.append(f"{y_str} / {x_str} = {f:.6f}")
    
    avg_factor = np.mean(factors)
    steps.append(f"")
    steps.append(f"Average conversion factor ≈ {avg_factor:.6f}")
    steps.append(f"")
    steps.append(f"Applying to {x_query} m:")
    steps.append(f"{x_query} × {avg_factor:.6f} = {avg_factor * x_query:.2f}")
    steps.append(f"")
    steps.append(f"\\boxed{{{answer}}}")
    
    return '\n'.join(steps)

# Test
row = train_df[train_df['category'] == 'unit_conversion'].iloc[0]
print(cot_unit_conversion(row['prompt'], row['answer']))

I need to convert 25.09 m using the secret conversion rule.

Let me find the conversion factor from the examples:

6.69 / 10.08 = 0.663690
11.83 / 17.83 = 0.663489
23.79 / 35.85 = 0.663598
11.32 / 17.06 = 0.663540
20.93 / 31.54 = 0.663602

Average conversion factor ≈ 0.663584

Applying to 25.09 m:
25.09 × 0.663584 = 16.65

\boxed{16.65}


### 1.4 Text Encryption

In [5]:
def cot_text_encryption(prompt, answer):
    """Generate CoT for substitution cipher decryption."""
    lines = prompt.strip().split('\n')
    examples = []
    query_text = None
    for line in lines:
        line = line.strip()
        if ' -> ' in line and 'encryption' not in line.lower():
            parts = line.split(' -> ')
            if len(parts) == 2:
                examples.append((parts[0].strip(), parts[1].strip()))
        elif 'decrypt the following text:' in line.lower():
            m = re.search(r'decrypt the following text:\s*(.*)', line, re.IGNORECASE)
            if m: query_text = m.group(1).strip()

    if not examples or not query_text:
        return None

    # Build mapping
    char_map = {}
    for enc, dec in examples:
        if len(enc) == len(dec):
            for e, d in zip(enc, dec):
                if e != ' ':
                    char_map[e] = d

    steps = []
    steps.append(f"This is a substitution cipher. I need to build a letter mapping from the examples.")
    steps.append(f"")
    steps.append(f"Building the substitution table by aligning encrypted and decrypted text:")
    steps.append(f"")
    
    # Show a subset of the mapping
    sorted_map = sorted(char_map.items())
    map_str = ', '.join(f'{e}→{d}' for e, d in sorted_map)
    steps.append(f"Mapping: {map_str}")
    steps.append(f"")
    
    steps.append(f"Decrypting: {query_text}")
    steps.append(f"")
    
    # Show word-by-word decryption
    enc_words = query_text.split()
    dec_words = answer.split()
    for i, enc_word in enumerate(enc_words):
        if i < len(dec_words):
            steps.append(f"'{enc_word}' → '{dec_words[i]}'")
    
    steps.append(f"")
    steps.append(f"\\boxed{{{answer}}}")
    
    return '\n'.join(steps)

# Test
row = train_df[train_df['category'] == 'text_encryption'].iloc[0]
print(cot_text_encryption(row['prompt'], row['answer']))

This is a substitution cipher. I need to build a letter mapping from the examples.

Building the substitution table by aligning encrypted and decrypted text:

Mapping: b→t, c→u, d→f, e→y, f→o, g→s, i→w, j→l, n→p, o→e, p→d, q→r, r→a, s→g, t→c, u→q, v→n, w→i, x→h, y→v, z→m

Decrypting: trb wzrswvog hffk

'trb' → 'cat'
'wzrswvog' → 'imagines'
'hffk' → 'book'

\boxed{cat imagines book}


### 1.5 Bit Manipulation

In [6]:
def cot_bit_manipulation(prompt, answer):
    """Generate CoT for bit manipulation puzzles."""
    examples = re.findall(r'([01]{8}) -> ([01]{8})', prompt)
    query_match = re.search(r'determine the output for:\s*([01]{8})', prompt)
    if not examples or not query_match:
        return None

    query = query_match.group(1)
    
    # Try to identify the actual rule for better CoT
    inputs = np.array([[int(b) for b in inp] for inp, _ in examples])
    outputs = np.array([[int(b) for b in out] for _, out in examples])
    
    # Check if bit permutation with optional inversion
    perm = []
    for op in range(8):
        found = False
        for ip in range(8):
            if np.array_equal(outputs[:, op], inputs[:, ip]):
                perm.append((ip, False))
                found = True
                break
            if np.array_equal(outputs[:, op], 1 - inputs[:, ip]):
                perm.append((ip, True))
                found = True
                break
        if not found:
            perm = None
            break
    
    steps = []
    steps.append(f"I need to find the transformation rule from these input→output examples and apply it to {query}.")
    steps.append(f"")
    
    if perm:
        steps.append(f"Analyzing bit positions to find the mapping:")
        steps.append(f"")
        for out_pos, (in_pos, inverted) in enumerate(perm):
            inv_str = " (inverted)" if inverted else ""
            steps.append(f"Output bit {out_pos} ← Input bit {in_pos}{inv_str}")
        steps.append(f"")
        steps.append(f"Applying to {query}:")
        result_bits = []
        for out_pos, (in_pos, inverted) in enumerate(perm):
            bit = int(query[in_pos])
            if inverted:
                bit = 1 - bit
            result_bits.append(str(bit))
        steps.append(f"Result: {''.join(result_bits)}")
    else:
        # Generic CoT when we can't determine the exact rule
        steps.append(f"Let me analyze the examples to identify the pattern:")
        steps.append(f"")
        for inp, out in examples[:4]:
            steps.append(f"{inp} → {out}")
        steps.append(f"")
        steps.append(f"Studying the bit-by-bit transformation pattern across all examples,")
        steps.append(f"I can identify the rule and apply it to the input {query}.")
    
    steps.append(f"")
    steps.append(f"\\boxed{{{answer}}}")
    
    return '\n'.join(steps)

# Test
row = train_df[train_df['category'] == 'bit_manipulation'].iloc[0]
print(cot_bit_manipulation(row['prompt'], row['answer']))

I need to find the transformation rule from these input→output examples and apply it to 00110100.

Let me analyze the examples to identify the pattern:

01010001 → 11011101
00001001 → 01101101
00010101 → 01010101
11111111 → 10000001

Studying the bit-by-bit transformation pattern across all examples,
I can identify the rule and apply it to the input 00110100.

\boxed{10010111}


### 1.6 Transformation Rules

In [7]:
def cot_transformation_rules(prompt, answer):
    """Generate CoT for transformation rule puzzles."""
    lines = prompt.strip().split('\n')
    examples = []
    query = None
    for line in lines:
        line = line.strip()
        if 'determine the result for:' in line.lower():
            m = re.search(r'determine the result for:\s*(.*)', line, re.IGNORECASE)
            if m: query = m.group(1).strip()
        elif ' = ' in line and 'transformation' not in line.lower() and 'below' not in line.lower():
            parts = line.split(' = ')
            if len(parts) == 2:
                examples.append((parts[0].strip(), parts[1].strip()))

    if not examples or not query:
        return None

    steps = []
    steps.append(f"I need to find the transformation rule from the examples and apply it to: {query}")
    steps.append(f"")
    steps.append(f"Analyzing the examples:")
    steps.append(f"")
    
    for lhs, rhs in examples:
        steps.append(f"{lhs} = {rhs}")
    
    steps.append(f"")
    
    # Check if numeric with operators
    numeric_pattern = re.compile(r'^(\d+)([^\d])(\d+)$')
    all_numeric = all(numeric_pattern.match(lhs) for lhs, _ in examples)
    
    if all_numeric:
        steps.append(f"These appear to be arithmetic operations with symbolic operators.")
        steps.append(f"")
        
        # Group by operator
        op_examples = defaultdict(list)
        for lhs, rhs in examples:
            m = numeric_pattern.match(lhs)
            if m:
                a, op, b = int(m.group(1)), m.group(2), int(m.group(3))
                op_examples[op].append((a, b, rhs))
        
        for op, cases in op_examples.items():
            steps.append(f"Operator '{op}':")
            for a, b, result in cases:
                # Try to identify what the operation does
                for op_name, func in [
                    ('addition', lambda a, b: str(a + b)),
                    ('subtraction', lambda a, b: str(a - b)),
                    ('multiplication', lambda a, b: str(a * b)),
                    ('concatenation', lambda a, b: str(a) + str(b)),
                    ('division', lambda a, b: str(a // b) if b != 0 else None),
                    ('modulo', lambda a, b: str(a % b) if b != 0 else None),
                    ('reverse subtraction', lambda a, b: str(b - a)),
                ]:
                    if func(a, b) == result:
                        steps.append(f"  {a} {op} {b} = {result} (this is {op_name}: {a} {op_name.split()[0][:3]}. {b} = {result})")
                        break
                else:
                    steps.append(f"  {a} {op} {b} = {result}")
    else:
        steps.append(f"Analyzing the character-level transformations to identify the pattern.")
        steps.append(f"Each input transforms according to a consistent rule.")
    
    steps.append(f"")
    steps.append(f"Applying the identified rule to {query}:")
    steps.append(f"")
    steps.append(f"\\boxed{{{answer}}}")
    
    return '\n'.join(steps)

# Test
row = train_df[train_df['category'] == 'transformation_rules'].iloc[2]
print(cot_transformation_rules(row['prompt'], row['answer']))

I need to find the transformation rule from the examples and apply it to: 69/52

Analyzing the examples:

34/44 = 1
41/32 = 9
34|25 = 69
87\64 = 8853

These appear to be arithmetic operations with symbolic operators.

Operator '/':
  34 / 44 = 1
  41 / 32 = 9 (this is subtraction: 41 sub. 32 = 9)
Operator '|':
  34 | 25 = 69
Operator '\':
  87 \ 64 = 8853

Applying the identified rule to 69/52:

\boxed{17/}


## 2. Generate CoT for All Training Data

In [8]:
# Master CoT generator
def generate_cot(row):
    prompt = row['prompt']
    answer = str(row['answer'])
    category = row['category']
    
    generators = {
        'numeral_system': cot_numeral_system,
        'gravitational': cot_gravitational,
        'unit_conversion': cot_unit_conversion,
        'text_encryption': cot_text_encryption,
        'bit_manipulation': cot_bit_manipulation,
        'transformation_rules': cot_transformation_rules,
    }
    
    gen = generators.get(category)
    if gen:
        return gen(prompt, answer)
    return None

# Generate CoT for all puzzles
print("Generating CoT for all 9,500 puzzles...")
train_df['cot'] = train_df.apply(generate_cot, axis=1)

# Check success rate
success = train_df['cot'].notna()
print(f"\nCoT generated: {success.sum()}/{len(train_df)} ({success.mean()*100:.1f}%)")
print("\nBy category:")
for cat in train_df['category'].unique():
    sub = train_df[train_df['category'] == cat]
    s = sub['cot'].notna().sum()
    print(f"  {cat}: {s}/{len(sub)}")

Generating CoT for all 9,500 puzzles...

CoT generated: 9500/9500 (100.0%)

By category:
  bit_manipulation: 1602/1602
  text_encryption: 1576/1576
  numeral_system: 1576/1576
  unit_conversion: 1594/1594
  gravitational: 1597/1597
  transformation_rules: 1555/1555


## 3. Format Training Data for SFT

### Critical: Match the EXACT Evaluation Format

From analyzing the competition's metric code (`score()` function), we discovered:

1. **NO system prompt** — evaluation only sends `[{'role': 'user', 'content': ...}]`
2. **Evaluation appends** this text to every prompt:
   `\nPlease put your final answer inside \boxed{}. For example: \boxed{your answer}`
3. **`enable_thinking=True`** — model generates `<think>...</think>` blocks
4. **Numerical tolerance**: `rel_tol=1e-2` for numbers, exact match for binary strings

Our training format MUST mirror this:
- **No system prompt** (model won't see one at inference)
- **Include the boxed instruction** in the user message
- **`<think>` reasoning → `</think>` → `\boxed{answer}`** in assistant response

In [ ]:
# ==========================================================================
# FORMAT TRAINING DATA TO MATCH EVALUATION EXACTLY
#
# From the metric code (generate_predictions):
#   user_content = item.prompt + '\nPlease put your final answer inside `\\boxed{}`...'
#   prompt = tokenizer.apply_chat_template(
#       [{'role': 'user', 'content': user_content}],  # NO system prompt!
#       enable_thinking=True,
#   )
#
# So we train with:
#   - NO system prompt (evaluation doesn't send one)
#   - The boxed instruction APPENDED to user message
#   - <think> reasoning </think> \boxed{answer} in assistant
# ==========================================================================

# The exact suffix the evaluation code appends to every prompt
EVAL_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

def format_for_sft(row):
    """Format training example to EXACTLY match evaluation inference format."""
    # User message = original prompt + evaluation suffix (matching what vLLM sees)
    user_content = row['prompt'] + EVAL_SUFFIX
    
    # Assistant message = <think> reasoning </think> \boxed{answer}
    # Split CoT into reasoning and boxed answer
    cot = row['cot']
    boxed_match = re.search(r'(\\boxed\{.*?\})\s*$', cot)
    
    if boxed_match:
        reasoning = cot[:boxed_match.start()].strip()
        boxed_answer = boxed_match.group(1)
        assistant_content = f"<think>\n{reasoning}\n</think>\n{boxed_answer}"
    else:
        assistant_content = f"<think>\n{cot}\n</think>"
    
    return {
        'messages': [
            # NO system prompt — evaluation doesn't use one
            {'role': 'user', 'content': user_content},
            {'role': 'assistant', 'content': assistant_content},
        ]
    }

# Filter to rows with valid CoT
valid_df = train_df[train_df['cot'].notna()].copy()
print(f"Valid training examples: {len(valid_df)}")

# Format
sft_data = [format_for_sft(row) for _, row in valid_df.iterrows()]

# Save as JSONL
output_path = 'train_cot_v3_metric_aligned.jsonl'
with open(output_path, 'w') as f:
    for item in sft_data:
        f.write(json.dumps(item) + '\n')

print(f"Saved {len(sft_data)} examples to {output_path}")

# Show a sample
print("\n" + "="*70)
print("Sample training example (MATCHES EVALUATION FORMAT):")
print("="*70)
sample = sft_data[0]
print(f"Number of messages: {len(sample['messages'])} (NO system prompt)")
print(f"\n--- User ---")
print(sample['messages'][0]['content'][-200:])  # Show the end with suffix
print(f"\n--- Assistant ---")
print(sample['messages'][1]['content'][:500])

## 4. Also save a simple format (prompt + completion)

In [10]:
# Save as simple prompt-completion pairs (useful for some frameworks)
simple_data = []
for _, row in valid_df.iterrows():
    simple_data.append({
        'id': row['id'],
        'category': row['category'],
        'prompt': row['prompt'],
        'answer': str(row['answer']),
        'cot_response': row['cot']
    })

# Save as JSON
with open('train_cot_simple.json', 'w') as f:
    json.dump(simple_data, f, indent=2)

print(f"Saved {len(simple_data)} examples to train_cot_simple.json")

# Stats
cot_lengths = [len(item['cot_response']) for item in simple_data]
print(f"\nCoT response length stats:")
print(f"  Min: {min(cot_lengths)} chars")
print(f"  Max: {max(cot_lengths)} chars")
print(f"  Mean: {np.mean(cot_lengths):.0f} chars")
print(f"  Median: {np.median(cot_lengths):.0f} chars")

Saved 9500 examples to train_cot_simple.json

CoT response length stats:
  Min: 202 chars
  Max: 620 chars
  Mean: 366 chars
  Median: 379 chars


## 5. Verify \\boxed{} Format

In [11]:
# Verify all CoT responses end with \boxed{answer}
boxed_count = 0
for item in simple_data:
    if f"\\boxed{{{item['answer']}}}" in item['cot_response']:
        boxed_count += 1

print(f"Responses with correct \\boxed{{}} format: {boxed_count}/{len(simple_data)} ({boxed_count/len(simple_data)*100:.1f}%)")

Responses with correct \boxed{} format: 9500/9500 (100.0%)


## 6. Token Length Validation

**Critical**: The competition uses `max_model_len=8192`. Any training example exceeding this
will be truncated during inference. We must verify all examples fit within this budget.

The total token count includes: system prompt + user prompt + assistant CoT response.
Since we don't have the Nemotron tokenizer locally, we estimate using a 4 chars/token heuristic
and flag any examples that may be too long. When you load the model for training,
re-run this check with the actual tokenizer.

## Summary

- Generated CoT training data for all 9,500 puzzles
- Saved in both JSONL (chat format) and JSON (simple format)
- All responses include `\boxed{answer}` for evaluation compatibility
- Token length validated against `max_model_len=8192`

**Important for notebook 03:**
- Use the **Nemotron ChatML template** with `<think>` tags for reasoning
- Model ID: `nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16`
- Target **both** Transformer attention AND Mamba-2 layers for LoRA

In [ ]:
# Token length estimation (heuristic: ~4 chars per token for English text)
# This is conservative — actual Nemotron tokenizer may differ.
# Re-run with real tokenizer during training setup (see notebook 03).

MAX_MODEL_LEN = 8192
CHARS_PER_TOKEN = 3.5  # Conservative estimate

print("=== Token Length Validation ===\n")

# Load the JSONL we just saved
data_to_check = []
jsonl_file = 'train_cot2.0.jsonl' if os.path.exists('train_cot2.0.jsonl') else 'train_cot.jsonl'
with open(jsonl_file, 'r') as f:
    for line in f:
        data_to_check.append(json.loads(line))

estimated_tokens = []
for item in data_to_check:
    total_chars = sum(len(m['content']) for m in item['messages'])
    # Add ~50 tokens for chat template overhead (special tokens, role markers)
    est_tokens = int(total_chars / CHARS_PER_TOKEN) + 50
    estimated_tokens.append(est_tokens)

estimated_tokens = np.array(estimated_tokens)

print(f"Estimated token lengths (using {CHARS_PER_TOKEN} chars/token):")
print(f"  Min:    {estimated_tokens.min()}")
print(f"  Max:    {estimated_tokens.max()}")
print(f"  Mean:   {estimated_tokens.mean():.0f}")
print(f"  Median: {np.median(estimated_tokens):.0f}")
print(f"  P95:    {np.percentile(estimated_tokens, 95):.0f}")
print(f"  P99:    {np.percentile(estimated_tokens, 99):.0f}")

over_limit = (estimated_tokens > MAX_MODEL_LEN).sum()
print(f"\n  Exceeding {MAX_MODEL_LEN} tokens: {over_limit}/{len(estimated_tokens)}")

if over_limit > 0:
    print(f"\n  WARNING: {over_limit} examples may exceed max_model_len!")
    print(f"  These will be truncated during training/inference.")
    print(f"  Consider shortening the CoT for these examples.")
else:
    print(f"\n  All examples fit within {MAX_MODEL_LEN} token limit.")

import os
